In [1]:
import pandas as pd
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, 
                             recall_score, f1_score, matthews_corrcoef)

# Create the model directory if it doesn't exist
os.makedirs('model', exist_ok=True)

In [ ]:
import pandas as pd

# 1. Define the direct URL to the UCI raw data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"

# 2. Define the column names manually
columns = [
    'id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 
    'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 
    'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 
    'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 
    'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 
    'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 
    'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst'
]

# 3. Read directly into a DataFrame!
df = pd.read_csv(url, names=columns)

# 4. Clean data (Drop the ID column)
df_cleaned = df.drop(columns=['id'], errors='ignore')

# 5. Convert target to binary (Malignant = 1, Benign = 0)
df_cleaned['diagnosis'] = df_cleaned['diagnosis'].map({'M': 1, 'B': 0})

# 6. Split features and target
X = df_cleaned.drop(columns=['diagnosis'])
y = df_cleaned['diagnosis']

# 7. Train-test split (80% training, 20% testing)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")
print("Data successfully loaded directly from the web!")

# Recombine the test features and test labels
test_data = X_test.copy()
test_data['diagnosis'] = y_test

# Save it as a physical file in your project folder
test_data.to_csv('test_data.csv', index=False)
print("test_data.csv saved successfully!")

Training data shape: (455, 30)
Testing data shape: (114, 30)
Data successfully loaded directly from the web!
test_data.csv saved successfully!


In [4]:
# Initialize the models
models = {
    "Logistic Regression": LogisticRegression(max_iter=10000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(random_state=42)
}

# Dictionary to hold the trained models
trained_models = {}

# Train each model
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"{name} trained successfully.")

Logistic Regression trained successfully.
Decision Tree trained successfully.
K-Nearest Neighbors trained successfully.
Naive Bayes trained successfully.
Random Forest trained successfully.


In [ ]:
results = []

for name, model in trained_models.items():
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate probability for AUC 
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    # Store results for printing
    results.append({
        "Model": name, "Accuracy": acc, "AUC": auc, 
        "Precision": prec, "Recall": rec, "F1": f1, "MCC": mcc
    })
    
    # Save the model to the 'model/' directory
    filename = name.lower().replace(" ", "_").replace("-", "") + ".pkl"
    joblib.dump(model, f"model/{filename}")
    print(f"Saved: model/{filename}")

print("\n--- Evaluation Metrics for README.md ---")
results_df = pd.DataFrame(results)
# Formatting to 4 decimal places
display(results_df.style.format({
    "Accuracy": "{:.4f}", "AUC": "{:.4f}", "Precision": "{:.4f}", 
    "Recall": "{:.4f}", "F1": "{:.4f}", "MCC": "{:.4f}"
}))

Saved: model/logistic_regression.pkl
Saved: model/decision_tree.pkl
Saved: model/knearest_neighbors.pkl
Saved: model/naive_bayes.pkl
Saved: model/random_forest.pkl

--- Evaluation Metrics for README.md ---


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9561,0.9977,0.9750,0.9070,0.9398,0.9068
1,Decision Tree,0.9474,0.9440,0.9302,0.9302,0.9302,0.8880
2,K-Nearest Neighbors,0.9561,0.9959,1.0000,0.8837,0.9383,0.9086
3,Naive Bayes,0.9737,0.9984,1.0000,0.9302,0.9639,0.9447
4,Random Forest,0.9649,0.9953,0.9756,0.9302,0.9524,0.9253
